In [1]:
!nvidia-smi
!apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!pip install -q fastapi uvicorn httpx

import os, subprocess, time, requests

env = {**os.environ,
       "OLLAMA_HOST": "127.0.0.1:11434",   # chỉ nghe nội bộ, không mở ra ngoài
       "OLLAMA_NUM_PARALLEL": "3",
       "OLLAMA_FLASH_ATTENTION": "1",
       "OLLAMA_KV_CACHE_TYPE": "q8_0",
       "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(["ollama", "serve"], env=env,
                 stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434"); print("Ollama đã sẵn sàng"); break
    except Exception:
        time.sleep(1)

!ollama pull qwen2.5:14b-instruct-q4_K_M

Wed Nov 11 11:04:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import requests
r = requests.post("http://127.0.0.1:11434/api/chat", timeout=300, json={
    "model": "qwen2.5:14b-instruct-q4_K_M",
    "messages": [{"role": "user", "content": "Hãy sửa chính tả: He do homework."}],
    "stream": False,
    "options": {"num_ctx": 4096, "temperature": 0},
})
print(r.json()["message"]["content"])

In [ ]:
!nvidia-smi

In [ ]:
import os, re, secrets, subprocess, time

TOKEN = secrets.token_urlsafe(24)
open("proxy.py", "w").write('''
import os, httpx
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse

TOKEN = os.environ["WORKER_TOKEN"]
app = FastAPI()

@app.get("/health")
async def health():
    return {"ok": True}

@app.post("/api/chat")
async def chat(req: Request):
    if req.headers.get("authorization") != f"Bearer {TOKEN}":
        raise HTTPException(401, "unauthorized")
    body = await req.json()
    body["stream"] = False
    async with httpx.AsyncClient(timeout=180) as c:
        r = await c.post("http://127.0.0.1:11434/api/chat", json=body)
    return JSONResponse(r.json(), status_code=r.status_code)
''')

subprocess.Popen(["uvicorn", "proxy:app", "--host", "127.0.0.1", "--port", "8000"],
                 env={**os.environ, "WORKER_TOKEN": TOKEN},
                 stdout=open("proxy.log", "w"), stderr=subprocess.STDOUT)
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                 stdout=open("cf.log", "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cf.log").read())
    if m:
        url = m.group(0); break

print("URL  :", url)
print("TOKEN:", TOKEN)